# Analysis – Shock-Observation Responses
**Michon Linde et al., Nature Communications**  
*"The Intermediate Hippocampus Integrates Shock-Observation and Spatial Information during Observational Fear Memory"*

Covers: **Fig. 2c–d** · **Fig. 3c** · **Extended Data Fig. 3a–b**

> Set `Folder_path` below to the directory containing the summary data tables.

## Imports

In [ ]:
import os
import numpy as np
import scipy as sc
import pandas as pd

from scipy.stats import chi2_contingency
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pingouin as pg

import seaborn as sns
from matplotlib import pyplot as plt

## Helper functions

In [ ]:
def compute_MC_pvalue(score, random_scores, method='two-sided'):
    """
    Two-sided Monte Carlo p-value for the observed proportion of responsive neurons
    against a null distribution obtained from circular firing-rate permutations.

    Parameters
    ----------
    score         : float — observed proportion
    random_scores : 1-D array — null distribution of proportions
    method        : 'two-sided' (default), 'greater', or 'lesser'

    Returns
    -------
    pval : float
    """
    pval_up   = (np.sum(random_scores >= np.max(score)) + 1) / (random_scores.shape[0] + 1)
    pval_down = (np.sum(random_scores <= np.min(score)) + 1) / (random_scores.shape[0] + 1)

    if method == 'greater':
        return pval_up
    if method == 'lesser':
        return pval_down
    # two-sided
    return np.min([pval_up, pval_down]) * 2

## Data loading

In [ ]:
# ── Set this path to the folder containing the summary data tables ──────────
Folder_path = "/data07/Fred/Ctx_Hpc/summaries/NatCom/"

# Single-unit shock-observation responses: one row per pyramidal neuron or interneuron
df_sum = pd.read_parquet(os.path.join(Folder_path, 'table_unit_ShockObs.parquet'))

# Trial-level place × shock conjunctive coding (used in Fig. 3c and Ext. Data Fig. 3b)
df_shock_place = pd.read_parquet(os.path.join(Folder_path, 'table_place_shock_coding.parquet'))

# Population-level shock-observation discriminability (AUC, Fig. 2d)
df_shock_behav = pd.read_parquet(os.path.join(Folder_path, 'table_place_shock_SVM.parquet'))

print("Loaded summary tables:")
print(f"  df_sum         : {df_sum.shape[0]} units — shock-observation responses")
print(f"  df_shock_place : {df_shock_place.shape[0]} rows — place × shock coding (per trial)")
print(f"  df_shock_behav : {df_shock_behav.shape[0]} rows — population AUC discriminability")
print()
print("Pyramidal neurons per subregion:")
print(df_sum.query("neuron_type == 'pyr'").groupby('pole').size().to_string())

---
## Figure 2c
**Putative pyramidal neurons recruited by shock observation across hippocampal subregions.**  
Heatmap of normalised mean firing rates aligned to footshock onset (0–1 s, gold dashed lines),  
separately for the dorsal, intermediate and ventral hippocampus.  
Neurons are sorted within each subregion: excited (top), inhibited, unresponsive (bottom).

In [ ]:
gap   = 12   # blank rows between subregions
count = 0

rates    = []
units_N  = []
units_idx = []

for po in ['dorsal', 'intermediate', 'ventral']:
    tmp = (df_sum
           .set_index(['rat', 'cluster'])
           .query("firing_rate > 0.1 and pole == '{}' and neuron_type == 'pyr'".format(po))
           .sort_values('shocks_response'))

    t = np.array(tmp['time'].iloc[0])

    rates.append(np.nan_to_num(np.vstack([r / np.max(r) for r in tmp['rates_shocks']])))
    rates.append(np.full((gap, len(t)), np.nan))
    units_N.append([1, len(tmp) - 1])
    units_idx.append([count, count + len(tmp)])
    count += len(tmp) + gap

fig, ax = plt.subplots(1, 1, figsize=(4, 8))
sns.heatmap(np.vstack(rates)[:-gap], cmap='viridis', cbar=None, ax=ax)

xlabels = [-2, -1, 0, 1, 2, 3]
xticks  = np.array([np.argmin(np.abs(t - ts)) for ts in xlabels])

ax.axvline(np.argmin(np.abs(t)),     color='orange', ls='--')  # shock onset
ax.axvline(np.argmin(np.abs(t - 1)), color='orange', ls='--')  # shock offset

ax.set(xlim=(xticks[0], xticks[-1]),
       ylabel='unit #', xlabel='time from shock onset (s)')
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels, rotation=0)
ax.set_yticks(np.concatenate(units_idx))
ax.set_yticklabels(np.concatenate(units_N), rotation=0)
ax.tick_params(axis='y', labelsize=8)

### Statistics — Fig. 2c | Monte Carlo test: proportion of shock-responsive pyramidal neurons per subregion

In [ ]:
# Compare observed proportions of excited/inhibited pyramidal neurons per subregion
# against null distributions from 1000 circular firing-rate permutations.
pval_th = 0.05

df_shuffles = {k: [] for k in ['subregion', 'neuron_type', 'stimulus',
                                 'response_type', 'proportion', 'shuffles',
                                 'pval', 'context_learning']}

RepType = {'excited': '>0.01', 'inhibited': '<-0.01'}

for po in ['dorsal', 'intermediate', 'ventral']:
    for Ntype in ['pyr']:
        tmp = df_sum.query("firing_rate > 0.1 and pole == '{}' and neuron_type == '{}'".format(po, Ntype))

        for stim in ['shocks', 'ctrl', 'immo']:
            for rt, criterion in RepType.items():
                prop          = np.nan
                shuffles_prop = np.nan
                pval          = np.nan

                if len(tmp) != 0:
                    prop = (len(tmp.query("p_{} < {} and response_{}{}".format(
                                stim, pval_th, stim, criterion))) / len(tmp))
                    shuffles = np.vstack(tmp['shuffled_p_{}'.format(stim)])
                    shuffles_prop = np.array([
                        np.sum(shuffles[:, i] < pval_th) / shuffles.shape[0]
                        for i in range(shuffles.shape[1])])
                    pval = compute_MC_pvalue(prop, shuffles_prop, method='greater')

                df_shuffles['subregion'].append(po)
                df_shuffles['neuron_type'].append(Ntype)
                df_shuffles['stimulus'].append(stim)
                df_shuffles['response_type'].append(rt)
                df_shuffles['proportion'].append(prop)
                df_shuffles['shuffles'].append(shuffles_prop)
                df_shuffles['pval'].append(pval)
                df_shuffles['context_learning'].append(np.nan)

df_shuffles = pd.DataFrame(df_shuffles)

print("Fig. 2c — Monte Carlo p-values for pyramidal neuron shock-observation responses")
print()
print(df_shuffles.query("stimulus == 'shocks'")[
    ['subregion', 'response_type', 'proportion', 'pval']].to_string(index=False))

# χ² test across subregions (dorsal / intermediate / ventral)
ct = pd.crosstab(
    df_sum.query("neuron_type == 'pyr' and firing_rate > 0.1")['pole'],
    df_sum.query("neuron_type == 'pyr' and firing_rate > 0.1")['shocks_response'])
chi2, pchi, dof, _ = chi2_contingency(ct)
print()
print("Fig. 2c — χ²({}) = {:.2f}, p = {:.4f}".format(dof, chi2, pchi))

---
## Figure 2d
**Population discriminability of shock-observation moments.**  
AUC of a linear classifier trained to separate speed- and proximity-corrected  
shock responses from matched control moments, separately for the full hippocampus (Hpc)  
and the dorsal (D), intermediate (I) and ventral (V) subregions,  
split by RECALLER and NON-RECALLER animals.

In [ ]:
var = 'AUC'

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.axhline(0.5, color='grey', ls='--', label='chance')

sns.boxplot(x='pole', hue='context_learning', y=var, data=df_shock_behav,
            order=['all', 'dorsal', 'intermediate', 'ventral'],
            hue_order=['not learned', 'learned'],
            palette=['grey', 'orchid'],
            boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax)
sns.stripplot(x='pole', hue='context_learning', y=var, data=df_shock_behav,
              order=['all', 'dorsal', 'intermediate', 'ventral'],
              hue_order=['not learned', 'learned'],
              palette=['grey', 'orchid'],
              dodge=True, size=8, alpha=0.6, ax=ax)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['NON-RECALLER', 'RECALLER'], frameon=False)
ax.set(ylim=(0, 1),
       xticklabels=['Hpc', 'D', 'I', 'V'],
       xlabel='', ylabel='AUC')
sns.despine(offset=True, trim=True)

### Statistics — Fig. 2d | AUC vs. chance (one-sample t-test) and RECALLER × subregion comparisons

In [ ]:
# One-sample t-test vs. 0.5 (chance) per subregion; RECALLER vs. NON-RECALLER comparison
for area in np.unique(df_shock_behav.pole):
    results_chance = pg.ttest(
        x=df_shock_behav.query("pole == '{}' and condition == 'shock'".format(area))['AUC'],
        y=0.5)
    results_group  = pg.ttest(
        df_shock_behav.query("pole == '{}' and context_learning == 'not learned' and condition == 'shock'".format(area))['AUC'],
        df_shock_behav.query("pole == '{}' and context_learning == 'learned' and condition == 'shock'".format(area))['AUC'],
        paired=False)
    print("  Subregion: {}".format(area))
    print("  AUC vs. chance (0.5):")
    print(results_chance.to_string())
    print("  NON-RECALLER vs. RECALLER:")
    print(results_group.to_string())
    print()

# Two-way ANOVA and post-hoc tests: AUC ~ subregion × recall status
var, factor1, factor2 = 'AUC', 'pole', 'context_learning'
data = df_shock_behav[[var, factor1, factor2]].copy()
data['group_combined'] = data[factor1] + '_' + data[factor2]

results  = pg.anova(data=data, dv=var, between=[factor1, factor2], ss_type=2)
posthoc1 = pg.pairwise_tests(dv=var, between=[factor1, factor2],
                              padjust='holm', effsize='cohen', data=data)

print("Fig. 2d — Two-way ANOVA: AUC ~ subregion × recall status")
print(results.to_string())
print()
print(posthoc1.to_string())

---
## Figure 3c
**Place-field-dependent gain modulation of shock-observation responses.**  
Linear regression of the corrected shock-observation response (speed- and proximity-corrected  
Δ firing rate) on the normalised baseline spatial firing rate at the observer's position,  
separately for footshock delivery (gold/orange) and matched control moments (sienna),  
for each hippocampal subregion. Each point is one trial.

In [ ]:
for po in ['dorsal', 'intermediate', 'ventral']:
    tmp_model = df_shock_place.query("neuron_type == 'pyr' and pole == '{}'".format(po))

    fig, ax = plt.subplots(1, 1, figsize=(3, 4))

    sns.regplot(x='base_place_rate_post', y='rate_delta_corr',
                data=tmp_model.query("stimulus == 'noshock'"),
                ci=95, marker='.', scatter_kws={'alpha': 0.1, 's': 80},
                color='sienna', label='control moments', ax=ax)
    sns.regplot(x='base_place_rate_post', y='rate_delta_corr',
                data=tmp_model.query("stimulus == 'shock'"),
                ci=95, marker='^', scatter_kws={'alpha': 0.1, 's': 30},
                color='darkorange', label='shock observation', ax=ax)

    ax.set(xlim=(-0.01, 1.01), ylim=(-0.2, 0.2),
           xlabel='normalised baseline spatial firing rate',
           ylabel=u'corrected Δrate (shock obs.)',
           title='{} hippocampus'.format(po))
    sns.despine(offset=True, trim=True)

### Statistics — Fig. 3c | Linear mixed model: Δrate ~ spatial firing × stimulus (shock vs. control)

In [ ]:
# Linear mixed model: corrected rate change ~ spatial firing × stimulus (shock vs control)
# Random effect: rat identity (intercept)
# Tested separately per subregion, and for effect of recall status.
stats = []
stats.append("=== Fig. 3c — Linear mixed model: pyramidal neurons ===")

for pole in ['dorsal', 'intermediate', 'ventral']:
    tmp_model = (df_shock_place
                 .groupby(['rat', 'cluster', 'trial', 'stimulus']).first()
                 .reset_index()
                 .query("neuron_type == 'pyr' and pole == '{}'".format(pole)))
    tmp_model = tmp_model.replace({'not learned': 'NON-RECALLER', 'learned': 'RECALLER'})
    tmp_model['response'] = ['ShockObs+' if r == 'excited' else 'rest'
                              for r in tmp_model.shocks_response]
    tmp_model['response'] = (tmp_model['response']
                              .astype('category')
                              .cat.reorder_categories(['rest', 'ShockObs+']))

    fixed_formula = "rate_delta_corr ~ base_place_rate_post * C(stimulus) * context_learning"
    vc_formula    = {'rat': 'C(rat)'}
    oo = np.ones(tmp_model.shape[0])

    model   = sm.MixedLM.from_formula(fixed_formula, data=tmp_model,
                                       groups=oo, vc_formula=vc_formula)
    results = model.fit()
    summ    = results.summary()

    stats.append("")
    stats.append("--- {} hippocampus ---".format(pole))
    stats.append(summ.tables[0].to_string())
    stats.append(summ.tables[1].to_string())

for line in stats:
    print(line)

---
## Extended Data Figure 3a
**Putative interneurons recruited by shock observation across hippocampal subregions.**  
Heatmap of normalised mean firing rates aligned to footshock onset (0–1 s, gold dashed lines),  
separately for the dorsal, intermediate and ventral hippocampus.  
Neurons are sorted within each subregion: excited (top), inhibited, unresponsive (bottom).

In [ ]:
gap   = 8
count = 0

rates_int  = []
units_N    = []
units_idx  = []

for po in ['dorsal', 'intermediate', 'ventral']:
    tmp = (df_sum
           .set_index(['rat', 'cluster'])
           .query("firing_rate > 0.1 and pole == '{}' and neuron_type == 'int'".format(po))
           .sort_values('shocks_response'))

    t = np.array(tmp['time'].iloc[0])

    rates_int.append(np.nan_to_num(np.vstack([r / np.max(r) for r in tmp['rates_shocks']])))
    rates_int.append(np.full((gap, len(t)), np.nan))
    units_N.append([1, len(tmp) - 1])
    units_idx.append([count, count + len(tmp)])
    count += len(tmp) + gap

fig, ax = plt.subplots(1, 1, figsize=(4, 8))
sns.heatmap(np.vstack(rates_int)[:-gap], cmap='viridis', cbar=None, ax=ax)

xlabels = [-2, -1, 0, 1, 2, 3]
xticks  = np.array([np.argmin(np.abs(t - ts)) for ts in xlabels])

ax.axvline(np.argmin(np.abs(t)),     color='orange', ls='--')
ax.axvline(np.argmin(np.abs(t - 1)), color='orange', ls='--')

ax.set(xlim=(xticks[0], xticks[-1]),
       ylabel='unit #', xlabel='time from shock onset (s)')
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels, rotation=0)
ax.set_yticks(np.concatenate(units_idx))
ax.set_yticklabels(np.concatenate(units_N), rotation=0)
ax.tick_params(axis='y', labelsize=8)

### Statistics — Extended Data Fig. 3a | Monte Carlo test: proportion of shock-responsive interneurons per subregion

In [ ]:
df_shuffles_int = {k: [] for k in ['subregion', 'neuron_type', 'stimulus',
                                    'response_type', 'proportion', 'shuffles',
                                    'pval', 'context_learning']}
RepType = {'excited': '>0.01', 'inhibited': '<-0.01'}

for po in ['dorsal', 'intermediate', 'ventral']:
    tmp = df_sum.query("firing_rate > 0.1 and pole == '{}' and neuron_type == 'int'".format(po))

    for stim in ['shocks', 'ctrl', 'immo']:
        for rt, criterion in RepType.items():
            prop          = np.nan
            shuffles_prop = np.nan
            pval          = np.nan

            if len(tmp) != 0:
                prop = (len(tmp.query("p_{} < {} and response_{}{}".format(
                            stim, pval_th, stim, criterion))) / len(tmp))
                shuffles = np.vstack(tmp['shuffled_p_{}'.format(stim)])
                shuffles_prop = np.array([
                    np.sum(shuffles[:, i] < pval_th) / shuffles.shape[0]
                    for i in range(shuffles.shape[1])])
                pval = compute_MC_pvalue(prop, shuffles_prop, method='greater')

            df_shuffles_int['subregion'].append(po)
            df_shuffles_int['neuron_type'].append('int')
            df_shuffles_int['stimulus'].append(stim)
            df_shuffles_int['response_type'].append(rt)
            df_shuffles_int['proportion'].append(prop)
            df_shuffles_int['shuffles'].append(shuffles_prop)
            df_shuffles_int['pval'].append(pval)
            df_shuffles_int['context_learning'].append(np.nan)

df_shuffles_int = pd.DataFrame(df_shuffles_int)
print("Ext. Data Fig. 3a — Monte Carlo p-values for interneuron shock-observation responses")
print()
print(df_shuffles_int.query("stimulus == 'shocks'")[
    ['subregion', 'response_type', 'proportion', 'pval']].to_string(index=False))

---
## Extended Data Figure 3b
**Place-field-dependent modulation of shock-observation responses in putative interneurons.**  
Same analysis as Fig. 3c applied to putative interneurons.

In [ ]:
for po in ['dorsal', 'intermediate', 'ventral']:
    tmp_model = df_shock_place.query("neuron_type == 'int' and pole == '{}'".format(po))

    fig, ax = plt.subplots(1, 1, figsize=(3, 4))

    sns.regplot(x='base_place_rate_post', y='rate_delta_corr',
                data=tmp_model.query("stimulus == 'noshock'"),
                ci=95, marker='.', scatter_kws={'alpha': 0.1, 's': 80},
                color='sienna', label='control moments', ax=ax)
    sns.regplot(x='base_place_rate_post', y='rate_delta_corr',
                data=tmp_model.query("stimulus == 'shock'"),
                ci=95, marker='^', scatter_kws={'alpha': 0.1, 's': 30},
                color='darkorange', label='shock observation', ax=ax)

    ax.set(xlim=(-0.01, 1.01), ylim=(-0.2, 0.2),
           xlabel='normalised baseline spatial firing rate',
           ylabel=u'corrected Δrate (shock obs.)',
           title='{} hippocampus'.format(po))
    sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 3b | Linear mixed model: Δrate ~ spatial firing × stimulus (interneurons)

In [ ]:
stats = []
stats.append("=== Ext. Data Fig. 3b — Linear mixed model: interneurons ===")

for pole in ['dorsal', 'intermediate', 'ventral']:
    tmp_model = (df_shock_place
                 .groupby(['rat', 'cluster', 'trial', 'stimulus']).first()
                 .reset_index()
                 .query("neuron_type == 'int' and pole == '{}'".format(pole)))
    tmp_model = tmp_model.replace({'not learned': 'NON-RECALLER', 'learned': 'RECALLER'})
    tmp_model['response'] = ['ShockObs+' if r == 'excited' else 'rest'
                              for r in tmp_model.shocks_response]
    tmp_model['response'] = (tmp_model['response']
                              .astype('category')
                              .cat.reorder_categories(['rest', 'ShockObs+']))

    fixed_formula = "rate_delta_corr ~ base_place_rate_post * C(stimulus) * context_learning"
    vc_formula    = {'rat': 'C(rat)'}
    oo = np.ones(tmp_model.shape[0])

    model   = sm.MixedLM.from_formula(fixed_formula, data=tmp_model,
                                       groups=oo, vc_formula=vc_formula)
    results = model.fit()
    summ    = results.summary()

    stats.append("")
    stats.append("--- {} hippocampus ---".format(pole))
    stats.append(summ.tables[0].to_string())
    stats.append(summ.tables[1].to_string())

for line in stats:
    print(line)